# Phase 2: from raw files to a PostgreSQL star schema

Phase 1 gave us raw JSON files. Phase 2 turns them into clean, typed tables that the assistant's tools can query. This notebook runs real SQL against the database so you can see how the tables fit together. The full write-up is [`docs/PIPELINE.md`](../docs/PIPELINE.md) (generated).

**Key ideas**
- **Star schema:** a *fact* table holds events (one row per report, one per recall); *dimension* tables hold the things events are about (product code, manufacturer, date). Lists inside a report (devices, problem terms, narratives) get child tables, so each item is stored and counted exactly once.
- **Quarantine:** a record that breaks a rule is not silently dropped; it is kept with the rule it broke.
- **Idempotent load:** running the load twice gives exactly the same tables.
- **Pre-registered rules:** cleaning rules, the manufacturer map and link rules were committed before the first load, so they could not be tuned to the results.

In [1]:
import json, os
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(ROOT / '.env')
con = psycopg.connect(os.environ['DATABASE_URL'])
def sql(q, *args):
    cur = con.execute(q, args)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
LS = json.loads((ROOT / 'results/phase2/load_summary.json').read_text(encoding='utf-8'))
VER = json.loads((ROOT / 'results/phase2/verify_phase2.json').read_text(encoding='utf-8'))

## 1. What got loaded, and what was quarantined

In [2]:
sql("""SELECT 'fact_report' t, count(*) FROM core.fact_report UNION ALL
       SELECT 'report_text', count(*) FROM core.report_text UNION ALL
       SELECT 'report_problem', count(*) FROM core.report_problem UNION ALL
       SELECT 'fact_recall', count(*) FROM core.fact_recall UNION ALL
       SELECT 'report_recall_link', count(*) FROM core.report_recall_link UNION ALL
       SELECT 'quarantine', count(*) FROM core.quarantine""")

,t,count
0,quarantine,1
1,fact_recall,1626
2,fact_report,154193
3,report_problem,200993
4,report_recall_link,531554
5,report_text,410058


In [3]:
sql('SELECT source, record_key, rule, detail FROM core.quarantine')

,source,record_key,rule,detail
0,events_LWP.jsonl.gz,20466318,R5,no narrative text


The one quarantined report has no narrative text at all (rule R5). It is kept, with its reason, instead of disappearing.

## 2. One report, joined across the star

A single report touches six tables. The query below follows one of them through the joins (the narrative is shortened).

In [4]:
key = sql("SELECT min(mdr_report_key) k FROM core.fact_report WHERE product_code='MBH' AND event_type='Injury'").k[0]
sql("""SELECT r.mdr_report_key, r.product_code, p.device_name, m.raw_name, m.parent_company,
              r.date_received, r.event_type, r.brand_name, r.pma_pmn_number
       FROM core.fact_report r JOIN core.dim_product_code p USING (product_code)
       JOIN core.dim_manufacturer m USING (manufacturer_id) WHERE r.mdr_report_key = %s""", key).T

,0
mdr_report_key,13150637
product_code,MBH
device_name,"Prosthesis, Knee, Patello/Femorotibial, Semi-C..."
raw_name,STRYKER ORTHOPAEDICS-MAHWAH
parent_company,STRYKER
date_received,2022-01-04
event_type,Injury
brand_name,X3 TRIATHLON CS INSERT #5 9MM
pma_pmn_number,K172634


In [5]:
display(sql('SELECT term FROM core.report_problem WHERE mdr_report_key = %s', key))
sql('SELECT text_type, left(text, 300) AS text FROM core.report_text WHERE mdr_report_key = %s ORDER BY text_seq', key)

,term
0,Adverse Event Without Identified Device or Use...


,text_type,text
0,Additional Manufacturer Narrative,IT WAS NOTED THAT THE DEVICE IS NOT AVAILABLE ...
1,Description of Event or Problem,REVISED A TRIATHLON TIBIAL INSERT IN ORDER TO ...


## 3. Manufacturer names become parent companies

FDA writes the same company many ways. The registered rules clean the names and merge known brands (St. Jude Medical into Abbott, Howmedica into Stryker, ...).

In [6]:
sql("""SELECT parent_company, count(*) AS spellings, string_agg(raw_name, ' | ' ORDER BY raw_name) AS examples
       FROM core.dim_manufacturer GROUP BY 1 HAVING count(*) > 2 ORDER BY 2 DESC LIMIT 8""")

,parent_company,spellings,examples
0,ZIMMER BIOMET,35,BIOMET | BIOMET FRANCE S.A.R.L. | BIOMET ORTHO...
1,STRYKER,25,Howmedica Osteonics Corp. | HOWMEDICA OSTEONIC...
2,MEDTRONIC,24,MEDTRONIC | MEDTRONIC 18000 DEVONSHIRE STREET ...
3,DEPUY SYNTHES (JOHNSON & JOHNSON),22,DEPUY (IRELAND) | DEPUY (IRELAND)/ DEPUY SYNTH...
4,ABBOTT,17,"ABBOTT | ABBOTT / ST. JUDE MEDICAL, CARDIOLOGY..."
5,SMITH & NEPHEW,10,SMITH & NEPHEW | Smith & Nephew Inc | SMITH & ...
6,CONFORMIS,9,CONFORMIS | Conformis Inc | CONFORMIS INC | Co...
7,BOSTON SCIENTIFIC,9,BOSTON SCIENTIFIC | BOSTON SCIENTIFIC CORP | B...


In [7]:
sql("""SELECT r.product_code, m.parent_company, extract(year FROM r.date_received)::int AS year, count(*)
       FROM core.fact_report r JOIN core.dim_manufacturer m USING (manufacturer_id)
       WHERE r.product_code = 'LWP' GROUP BY 1, 2, 3 HAVING count(*) > 100 ORDER BY 2, 3""")

,product_code,parent_company,year,count
0,LWP,ABBOTT,2022,6432
1,LWP,ABBOTT,2023,4522
2,LWP,ABBOTT,2024,2469
3,LWP,BOSTON SCIENTIFIC,2022,7180
4,LWP,BOSTON SCIENTIFIC,2023,7028
5,LWP,BOSTON SCIENTIFIC,2024,7545
6,LWP,MEDTRONIC,2022,294
7,LWP,MEDTRONIC,2023,438
8,LWP,MEDTRONIC,2024,168
9,LWP,MICROPORT,2022,193


For pacemakers (LWP), Phase 1 showed St. Jude Medical and Abbott as two separate makers. After merging they are one parent, Abbott, which is how a quality engineer would think about it.

## 4. Linking reports to recalls

Two registered rules: **510(k)** (same clearance number and product code) and **firm_window** (same parent company and product code, report received 180 days before to 365 days after the recall started). A link is context, never proof.

In [8]:
sql("""SELECT link_type, count(*) AS links, count(DISTINCT mdr_report_key) AS reports,
              count(DISTINCT product_res_number) AS recalls,
              percentile_disc(0.5) WITHIN GROUP (ORDER BY days_from_initiation) AS median_days
       FROM core.report_recall_link GROUP BY 1""")

,link_type,links,reports,recalls,median_days
0,510k,333623,66734,944,903
1,firm_window,197931,64888,200,126


In [9]:
sql("""SELECT l.link_type, c.product_res_number, c.event_date_initiated, c.root_cause,
              left(c.reason_for_recall, 120) AS reason, l.days_from_initiation
       FROM core.report_recall_link l JOIN core.fact_recall c USING (product_res_number)
       WHERE l.mdr_report_key = (SELECT mdr_report_key FROM core.report_recall_link
                                 WHERE link_type = 'firm_window' ORDER BY mdr_report_key LIMIT 1)
       ORDER BY abs(l.days_from_initiation) LIMIT 5""")

,link_type,product_res_number,event_date_initiated,root_cause,reason,days_from_initiation
0,firm_window,Z-2017-2021,2021-06-03,Device Design,The dual chamber INGENIO family pacemakers or ...,212
1,firm_window,Z-2016-2021,2021-06-03,Device Design,The dual chamber INGENIO family pacemakers or ...,212
2,firm_window,Z-2015-2021,2021-06-03,Device Design,The dual chamber INGENIO family pacemakers or ...,212
3,firm_window,Z-2001-2021,2021-06-03,Nonconforming Material/Component,Latent release of small amounts of hydrogen wi...,212
4,firm_window,Z-2000-2021,2021-06-03,Nonconforming Material/Component,Latent release of small amounts of hydrogen wi...,212


## 5. Verification

`python -m mdqc.verify_phase2` rebuilds every table from the raw files with separate code and compares every row and column, plants faults, reloads, and reproduces Phase 1's numbers:

In [10]:
pd.DataFrame([(c['id'], c['check'], 'pass' if c['pass'] else 'FAIL') for c in VER['checks']],
             columns=['id', 'check', 'result']).sort_values('id')

,id,check,result
0,P1,"loaded + quarantined = raw records, every file",pass
9,P11,database = openFDA's own server-side counts (p...,pass
1,P2,every loaded row and column = independent rebu...,pass
2,P3,no personal columns; contact names/phones abse...,pass
6,P4,second load gives identical table fingerprints,pass
7,P5,each planted fault quarantined with its rule; ...,pass
3,P6,manufacturer clean/parent mapping = independen...,pass
4,P7,invalid recall dates repaired by the registere...,pass
5,P8,report-recall links = independent recomputation,pass
8,P9,SQL reproduces Phase 1 numbers (quarantined re...,pass


In [11]:
con.close()

## Why we did these tasks

| Task | Why we did it | What we learned | What it changes next |
|---|---|---|---|
| Separate database and login | Projects must not share tables or passwords | An existing admin connection can create a least-privilege login once | Every later phase connects as mdqc_app |
| Pre-registered rules before the first load | So cleaning could not be tuned to make numbers look good | The registered rules caught a real report with no narrative and repaired the 0013 recall date | Rule changes from now on are labelled post hoc |
| Star schema with child tables | Counts must be right by construction (e.g. a term once per report) | Repeated FDA lists (devices, terms, texts) belong in their own tables | Phase 3 and the assistant's tools query clean, typed tables |
| Dropped personal fields at load | Privacy by default; the assistant never needs them | Contact names and phones were in most reports | Nothing downstream can leak what was never stored |
| Manufacturer parents | Trends per company need one name per company | Hundreds of spellings became about a hundred parents; a small misspelled tail remains | Phase 3 signals are computed per parent company |
| Report-recall links | An investigator asks 'was there a recall for this?' | 510(k) links are broad in time; pacemakers can only link by firm and date | Phase 4 tools filter links by date and label their strength |
| Independent verifier, planted faults, tamper test | A pipeline that cannot fail its check proves nothing | Every row matched; every planted fault and a single edited cell were caught | CI runs the same checks on a real PostgreSQL for every push |